[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/35_bpe.ipynb)

# 🔴 Hard: Byte-Pair Encoding (BPE)

*Inference & Decoding*
Implement **Byte-Pair Encoding**: learn a merge table from a corpus, then use
it to tokenize new text.

### Signature
```python
class SimpleBPE:
    def __init__(self): ...                        # self.merges = []
    def train(self, corpus, num_merges): ...       # fills self.merges
    def encode(self, text): ...                    # -> list[str]
```

### The algorithm
1. Split every word into characters and append `'</w>'` as an end-of-word marker
2. Count each word's frequency in the corpus
3. Repeat `num_merges` times:
   - count every adjacent symbol pair, weighted by word frequency
   - take the **most frequent** pair, append it to `self.merges`
   - rewrite every word with that pair fused into one symbol
4. `encode` splits on whitespace, appends `'</w>'`, and replays `self.merges`
   **in training order**

### Rules
- Pure Python — no JAX needed here, and no `tokenizers`/`sentencepiece`
- Stop early if there are no pairs left
- `self.merges` is a list of `(a, b)` tuples in the order they were learned

### Why the merge ORDER is the whole model
The merges are not a set, they are a **sequence**. `('e','s')` learned before
`('es','t')` is what lets `est` form at all — apply them in a different order
and the second merge never matches. This is why a BPE tokenizer ships its merge
list as an ordered file, and why you cannot add a merge in the middle without
retraining.

### Why `'</w>'`
Without an end-of-word marker, `"est"` in *estimate* and `"est"` in *fastest*
are the same symbol, and the tokenizer cannot tell a prefix from a suffix. The
marker makes word-final position part of the token's identity.

### What BPE buys you
It sits between characters (no unknown tokens, but very long sequences) and
whole words (short sequences, but a huge vocabulary and an `<UNK>` problem for
anything unseen). BPE gets an open vocabulary — *any* string is encodable —
with sequences only a few times longer than word-level. The cost is that token
boundaries are a statistical artifact of the training corpus, which is exactly
why models are bad at character-level tasks like counting letters.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class SimpleBPE:
    """Byte-pair encoding: learn merges, then apply them."""

    def __init__(self):
        self.merges = []

    def train(self, corpus, num_merges):
        """Learn `num_merges` merges from `corpus` (a list of words).

        Fills self.merges with (a, b) tuples in the order learned.
        """
        pass  # Replace this

    def encode(self, text):
        """Tokenize `text` by replaying self.merges in order. -> list[str]"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
bpe = SimpleBPE()
corpus = ["low"] * 5 + ["lower"] * 2 + ["newest"] * 6 + ["widest"] * 3

bpe.train(corpus, num_merges=10)
print("learned merges, in order:")
for i, m in enumerate(bpe.merges):
    print(f"  {i}: {m}")

print("\nencode('newest'):", bpe.encode("newest"))
print("encode('lowest'):", bpe.encode("lowest"), "  <- never seen, still encodable")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("bpe")

# hint("bpe")      # stuck? nudge without the answer
# solution("bpe")  # spoiler: the reference implementation